# PatchTSMixer Forecasting

In [ ]:
import pandas as pd
import torch
from transformers import PatchTSMixerForPrediction
import matplotlib.pyplot as plt
import numpy as np

## Load Data

In [ ]:
df_hist = pd.read_parquet('../../data/hist.parquet')
df_future = pd.read_parquet('../../data/future_covariates.parquet')

df_hist['time_idx'] = pd.to_datetime(df_hist['time_idx'])
df_hist = df_hist.set_index('time_idx')

print("Historical Data:")
print(df_hist.head())

## Prepare Data for Model

In [ ]:
prediction_length = 12
context_length = 64 

target_series = df_hist[df_hist['item_id'] == 'T1']['value'].values
context = target_series[-context_length:]

past_values = torch.tensor(context, dtype=torch.float32).unsqueeze(0)


## Load Model and Predict

In [ ]:
model = PatchTSMixerForPrediction.from_pretrained("ibm/patchtsmixer-base-prediction-etth1")

with torch.no_grad():
    outputs = model.generate(
        past_values=past_values
    )
    forecast = outputs.sequences.numpy().squeeze()

## Visualize Results

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(target_series[-context_length:], label='Historical Data')
forecast_index = pd.to_datetime(df_hist[df_hist['item_id'] == 'T1'].index[-1]) + pd.to_timedelta(range(1, prediction_length + 1), unit='H')
plt.plot(forecast_index, forecast[-prediction_length:], label='Forecast', color='red')
plt.title('PatchTSMixer Forecast vs. Historical Data')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()